# 日本語の時間表現を抽出して正規化する (ja-timex)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_semantics/timex_ja.ipynb)

NLP クックブック 09-09 ／ 意味解析

「来週の金曜日」「3 日前」「午後 3 時半」といった表現を見つけて、計算機で扱える日付や時間の形式に直したい。

In [1]:
!pip install -q ja-timex

In [2]:
import pendulum
from ja_timex import TimexParser

text = "彼は2008年4月に入社し、週に3回、30分間のジョギングを10年以上続けている。来週の金曜日の午後3時半から会議があり、一昨日届いた資料を使う。"
parser = TimexParser(reference=pendulum.datetime(2026, 11, 14, tz="Asia/Tokyo"))   # 「来週」「一昨日」の基準になる日
for t in parser.parse(text):
    print(f"{t.text:10} {t.type:9} value={t.value:12} mod={t.mod or '-'}")

2008年4月    DATE      value=2008-04-XX   mod=-
週に3回       SET       value=P1W          mod=-
30分間       DURATION  value=PT30M        mod=-
10年        DURATION  value=P10Y         mod=-
来週         DURATION  value=P1W          mod=AFTER
金曜日        DATE      value=XXXX-WXX-5   mod=-
午後3時半      TIME      value=T15-30-XX    mod=-
一昨日        DURATION  value=P2D          mod=BEFORE


In [3]:
# TIMEX3 形式のタグとして出力 (TimeML。時間情報のアノテーションの標準形式)
for t in parser.parse("明日の朝9時に東京を出発して、3時間かけて京都に行く。"):
    print(t.to_tag())

<TIMEX3 tid="t0" type="DURATION" value="P1D" mod="AFTER">明日</TIMEX3>
<TIMEX3 tid="t1" type="TIME" value="T09-XX-XX">朝9時</TIMEX3>
<TIMEX3 tid="t2" type="DURATION" value="PT3H">3時間</TIMEX3>


- TIMEX3 の 4 つの型: DATE (日付)、TIME (時刻)、DURATION (期間の長さ)、SET (頻度・繰り返し)。
- 「来週」「一昨日」のような相対的な表現は、「基準から 1 週間後」「2 日前」という長さと向き (`mod`) で表される。具体的な日付にするには、文書の作成日などの基準日から自分で計算する。
- 「来週の金曜日」は「来週」と「金曜日」の 2 つに分かれる。複数の表現を組み合わせた解釈は自分で行う必要がある。
- 規則ベースなので高速で、挙動が予測できる。一方、「先の大戦」「バブルの頃」のような表現や、文脈に依存する表現 (「その翌日」) は扱えない。
- 英語には HeidelTime や SUTime、多言語には `dateparser` (文字列処理の章を参照) がある。